In [0]:
dbutils.secrets.listScopes()

In [0]:
# install requirements
%pip install pymongo

In [0]:
# COMMAND ----------
import pymongo
import certifi
from pyspark.sql.functions import current_timestamp, lit

# 1. Fetch connection string securely from Key Vault Scope
mongo_uri = dbutils.secrets.get(scope="captone_scope", key="mongodb-conn-string")

# 2. Establish Mongo Client connection
client = pymongo.MongoClient(
    mongo_uri,
    tls=True,
    tlsCAFile=certifi.where()
)

# 3. Connect to the 'SPT_Stores' Database
db = client["SPT_Stores"]

# 4. Helper Function to Ingest Collections to maven_test.bronze
def ingest_mongo_collection_to_bronze(collection_name, target_table_name):
    print(f"Reading collection '{collection_name}' from MongoDB 'SPT_Stores'...")
    
    collection = db[collection_name]
    # Fetch all documents excluding MongoDB internal '_id'
    documents = list(collection.find({}, {"_id": 0}))
    
    if not documents:
        print(f"No documents found in collection '{collection_name}'. Skipping...")
        return

    # Convert Python dictionaries to PySpark DataFrame & append governance metadata
    df_raw = spark.createDataFrame(documents)
    df_bronze = df_raw \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_system", lit("MONGODB_ATLAS"))
    
    # Save directly into maven_test catalog bronze schema
    target_table = f"maven_test.bronze.{target_table_name}"
    (df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )
    print(f"Successfully landed {df_bronze.count()} records into Delta table: '{target_table}'")

# COMMAND ----------
# 5. List available collections in SPT_Stores
collections = db.list_collection_names()
print(f"Available collections in SPT_Stores: {collections}")

# 6. Run ingestion for each collection (replace collection names below with your exact collection names)
for collection_name in collections:
    table_name = f"bronze_mongodb_{collection_name.lower()}"
    ingest_mongo_collection_to_bronze(collection_name, table_name)